# Day 05 — Pandas, Veri Hattı ve Veri Kalitesi
## Pandas ile Veri Ön İşleme, Eksik Değer Yönetimi ve Veri Kalitesi Değerlendirmesi

> **Aşama:** Faz 1 — Problem, Veri ve Geliştirme Temelleri (Day 01–08)
> **Resmi Staj Defteri Konusu:** Pandas, Veri Hattı ve Veri Kalitesi (Yaprak 9 & 10)

### 1. Problem
Sensör telemetri loglarında veya üretim çizelgelerinde eksik (null), hatalı biçimlendirilmiş veya fiziksel sınırların dışına çıkan aykırı değerler bulunur. Ham veriyi doğrudan analize almak makine öğrenmesi modellerinin hatalı sonuç üretmesine yol açar.

### 2. Why the Problem Matters
Pandas kütüphanesi, tabular verilerin filtrelenmesi, eksik verilerin tespiti ve veri boru hattında (ETL) dönüştürülmesi için endüstri standardıdır. Veri kalitesinin (tamlık, geçerlilik, tutarlılık) ölçülmesi verinin güvenilirliğini sağlar.

### 3. Engineering Concepts
- **Veri Kalitesi Boyutları**: Tamlık (Completeness), Geçerlilik (Validity), Tutarlılık (Consistency).
- **Karantina Ayrımı (Quarantine Split)**: Hatalı kayıtların temiz veri setinden ayrılarak loglanması.
- **İmputasyon (Imputation)**: Eksik verilerin medyan veya ortalama ile tamamlanması.

In [ ]:
# 4. Library / API Investigation
import pandas as pd
import numpy as np
print(f"Pandas Version: {pd.__version__}")

In [ ]:
# 5. Minimal Implementation
from typing import Dict, List, Tuple, Any
from pydantic import BaseModel
import pandas as pd
import numpy as np

class DataQualityReport(BaseModel):
    total_rows: int
    clean_rows: int
    quarantine_rows: int
    completeness_score_pct: float
    validity_score_pct: float

class PandasDataPipeline:
    def __init__(self):
        pass

    def clean_and_profile(
        self,
        df: pd.DataFrame,
        required_columns: List[str],
        range_rules: Dict[str, Tuple[float, float]]
    ) -> Tuple[pd.DataFrame, pd.DataFrame, DataQualityReport]:
        if df.empty:
            raise ValueError("Girdi veri çerçevesi boş olamaz.")
        
        # Eksik deger kontrolu
        valid_mask = df[required_columns].notnull().all(axis=1)
        
        # Sinir kurallari kontrolu
        for col, (min_v, max_v) in range_rules.items():
            if col in df.columns:
                valid_mask = valid_mask & (df[col] >= min_v) & (df[col] <= max_v)
        
        clean_df = df[valid_mask].copy()
        quarantine_df = df[~valid_mask].copy()
        
        total = len(df)
        clean_count = len(clean_df)
        comp = float(df[required_columns].notnull().mean().mean() * 100.0)
        valid = float((clean_count / total) * 100.0) if total > 0 else 0.0
        
        report = DataQualityReport(
            total_rows=total,
            clean_rows=clean_count,
            quarantine_rows=len(quarantine_df),
            completeness_score_pct=round(comp, 2),
            validity_score_pct=round(valid, 2)
        )
        return clean_df, quarantine_df, report

df = pd.DataFrame({
    "loom_id": ["L1", "L2", "L3", "L4", "L5"],
    "rpm": [820.0, np.nan, 840.0, 790.0, 810.0],
    "tension_cn": [410.0, 420.0, -10.0, 430.0, 405.0]
})

pipeline = PandasDataPipeline()
clean_df, quarantine_df, report = pipeline.clean_and_profile(
    df=df,
    required_columns=["loom_id", "rpm", "tension_cn"],
    range_rules={"rpm": (600, 1000), "tension_cn": (0, 800)}
)
print("Temiz Veri Adedi:", len(clean_df))
print("Karantina Veri Adedi:", len(quarantine_df))
print("Veri Kalite Skoru: %", report.validity_score_pct)


In [ ]:
# 6. Experiment: Missing Value Profiling
missing_summary = df.isnull().sum()
print("Eksik Değer Özeti:")
print(missing_summary)

In [ ]:
# 7. Visualization
import matplotlib.pyplot as plt

metrics = ["Tamlık (Completeness)", "Geçerlilik (Validity)"]
scores = [report.completeness_score_pct, report.validity_score_pct]

plt.figure(figsize=(5, 3.5))
plt.bar(metrics, scores, color=["#2ca02c", "#1f77b4"], width=0.4)
plt.ylim(0, 100)
plt.ylabel("Kalite Skoru (%)")
plt.title("Pandas Veri Hattı Kalite Raporu")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert len(clean_df) + len(quarantine_df) == len(df)
assert len(clean_df) == 3
print("Veri hattı ayrıştırması başarıyla doğrulandı.")

In [ ]:
# 9. Failure Cases: Empty input DataFrame
try:
    pipeline.clean_and_profile(pd.DataFrame(), [], {})
except ValueError as e:
    print("Beklenen boş DataFrame hatası yakalandı:", e)

### 10. Conclusions
Pandas kullanılarak endüstriyel üretim verilerinde eksik ve aralık dışı kayıtlar otomatik tespit edilmiş, karantina mekanizmasıyla modeller için güvenli bir veri boru hattı kurulmuştur.